# OOMD: SQL vs NoSQL Document Model Comparison

Use case: e-commerce order tracking. This notebook compares normalized relational tables with JSON documents using SQLite. Run all cells from top to bottom.

**Note:** This is not a direct benchmark of PostgreSQL/MySQL versus MongoDB.

In [ ]:
%pip install pandas matplotlib


In [ ]:
from pathlib import Path
import json, platform, sqlite3, statistics, time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = Path('results')
OUT.mkdir(exist_ok=True)
SIZES = [1000, 5000, 10000]
REPEATS = 5
STATUSES = ['pending', 'shipped', 'delivered', 'cancelled']
CITIES = ['Pune', 'Mumbai', 'Nashik', 'Nagpur']
print('Python:', platform.python_version(), '| SQLite:', sqlite3.sqlite_version)
print('JSON support:', sqlite3.connect(':memory:').execute("SELECT json_valid('{\"ok\": true}')").fetchone()[0])


## 1. Define the two data models and benchmark functions

In [ ]:
def build_and_populate(model, n):
    conn = sqlite3.connect(':memory:')
    conn.execute('PRAGMA journal_mode=MEMORY')
    conn.execute('PRAGMA synchronous=OFF')
    start = time.perf_counter()
    if model == 'SQL':
        conn.executescript('''CREATE TABLE users (user_id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT UNIQUE NOT NULL, city TEXT NOT NULL);
        CREATE TABLE orders (order_id INTEGER PRIMARY KEY, user_id INTEGER NOT NULL, order_date TEXT NOT NULL, status TEXT NOT NULL, total REAL NOT NULL, FOREIGN KEY(user_id) REFERENCES users(user_id));
        CREATE INDEX idx_orders_status ON orders(status);
        CREATE INDEX idx_orders_user ON orders(user_id);''')
        users = [(i, f'User {i}', f'user{i}@example.com', CITIES[i % 4]) for i in range(1, n+1)]
        orders = [(i, i, f'2026-09-{(i % 28)+1:02d}', STATUSES[i % 4], round(100+(i % 100)*13.37, 2)) for i in range(1, n+1)]
        conn.executemany('INSERT INTO users VALUES (?, ?, ?, ?)', users)
        conn.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?)', orders)
    elif model == 'Document':
        conn.execute('CREATE TABLE documents (doc_id INTEGER PRIMARY KEY, body TEXT NOT NULL)')
        docs = []
        for i in range(1, n+1):
            d = {'user_id': i, 'name': f'User {i}', 'email': f'user{i}@example.com', 'city': CITIES[i % 4],
                 'order': {'order_id': i, 'order_date': f'2026-09-{(i % 28)+1:02d}', 'status': STATUSES[i % 4], 'total': round(100+(i % 100)*13.37, 2)}}
            docs.append((i, json.dumps(d, separators=(',', ':'))))
        conn.executemany('INSERT INTO documents VALUES (?, ?)', docs)
        conn.execute("CREATE INDEX idx_documents_status ON documents(json_extract(body, '$.order.status'))")
    conn.commit()
    return conn, (time.perf_counter()-start)*1000

def run_query(conn, model):
    start = time.perf_counter()
    if model == 'SQL':
        rows = conn.execute("SELECT o.order_id, u.name, o.status, o.total FROM orders o JOIN users u ON u.user_id=o.user_id WHERE o.status='shipped'").fetchall()
    else:
        rows = conn.execute("SELECT json_extract(body,'$.order.order_id'), json_extract(body,'$.name'), json_extract(body,'$.order.status'), json_extract(body,'$.order.total') FROM documents WHERE json_extract(body,'$.order.status')='shipped'").fetchall()
    return (time.perf_counter()-start)*1000, len(rows)

def update_records(conn, model, n):
    count = max(1, n//100)
    start = time.perf_counter()
    if model == 'SQL':
        conn.execute("UPDATE orders SET status='delivered' WHERE order_id <= ?", (count,))
    else:
        conn.execute("UPDATE documents SET body=json_set(body,'$.order.status','delivered') WHERE doc_id <= ?", (count,))
    conn.commit()
    return (time.perf_counter()-start)*1000, count

def footprint_kb(conn):
    pages = conn.execute('PRAGMA page_count').fetchone()[0]
    size = conn.execute('PRAGMA page_size').fetchone()[0]
    return pages*size/1024

def benchmark_one(model, n, repeats=5):
    measurements = {k: [] for k in ['population_ms','query_ms','update_ms','footprint_kb']}
    matched = updated = None
    for _ in range(repeats):
        conn, pop = build_and_populate(model, n)
        query, matched = run_query(conn, model)
        update, updated = update_records(conn, model, n)
        measurements['population_ms'].append(pop)
        measurements['query_ms'].append(query)
        measurements['update_ms'].append(update)
        measurements['footprint_kb'].append(footprint_kb(conn))
        conn.close()
    result = {k: statistics.median(v) for k, v in measurements.items()}
    result.update({'records': n, 'model': model, 'repeats': repeats, 'matched_rows': matched, 'updated_records': updated})
    return result


## 2. Run the experiment

In [ ]:
results = []
for n in SIZES:
    for model in ['SQL', 'Document']:
        r = benchmark_one(model, n, REPEATS)
        results.append(r)
        print(f"Finished {model:8s} | {n:>6,} records | query {r['query_ms']:.3f} ms")
df = pd.DataFrame(results)
df.to_csv(OUT / 'performance_results.csv', index=False)
display(df.round(3))


## 3. Performance evaluation matrix

In [ ]:
matrix_rows = []
for n in SIZES:
    a = df[(df.records == n) & (df.model == 'SQL')].iloc[0]
    b = df[(df.records == n) & (df.model == 'Document')].iloc[0]
    for metric, label, unit in [('population_ms','Population time','ms'), ('query_ms','Status-filter query latency','ms'), ('update_ms','Update latency (~1% records)','ms'), ('footprint_kb','SQLite page footprint proxy','KB')]:
        av, bv = float(a[metric]), float(b[metric])
        if av < bv: obs = f'SQL lower by {(bv-av)/bv*100:.1f}%' if bv else 'SQL lower'
        elif bv < av: obs = f'Document lower by {(av-bv)/av*100:.1f}%' if av else 'Document lower'
        else: obs = 'Equal'
        matrix_rows.append({'Records': n, 'Metric': f'{label} ({unit})', 'Approach A - SQL': round(av,3), 'Approach B - Document': round(bv,3), 'Observation': obs})
matrix = pd.DataFrame(matrix_rows)
display(matrix)
matrix.to_csv(OUT / 'comparison_matrix.csv', index=False)


## 4. Generate graphs

In [ ]:
def plot_metric(metric, title, ylabel, filename):
    plt.figure(figsize=(8,5))
    for model, label in [('SQL','Relational SQL'), ('Document','JSON document model')]:
        subset = df[df.model == model].sort_values('records')
        plt.plot(subset.records, subset[metric], marker='o', linewidth=2, label=label)
    plt.title(title); plt.xlabel('Number of customer-order records'); plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
    plt.savefig(OUT / filename, dpi=180)
    plt.show()

plot_metric('query_ms', 'Status-filter query latency', 'Median elapsed time (ms)', 'query_latency.png')
plot_metric('population_ms', 'Dataset population time', 'Median elapsed time (ms)', 'population_time.png')
plot_metric('update_ms', 'Update latency for approximately 1% of records', 'Median elapsed time (ms)', 'update_latency.png')
plot_metric('footprint_kb', 'SQLite database page footprint proxy', 'Page footprint (KB)', 'database_footprint.png')


## 5. Interpretation and limitations

Use the measured matrix and graphs to answer the report questions. The lower value is better for latency and population time. The footprint is a SQLite page-count proxy, not process memory. The experiment uses in-memory SQLite and synthetic data; results are environment-specific. It is not a direct comparison of PostgreSQL/MySQL and MongoDB.